In [1]:
import sqlite3
import pandas as pd

df = pd.read_csv("brazilian_ecommerce_data.csv")

conn = sqlite3.connect("ecommerce.db")
df.to_sql("raw_ecommerce_data", conn, if_exists="replace", index=False)
conn.close()

print("Success! 'ecommerce.db' has been created.")

Success! 'ecommerce.db' has been created.


In [2]:
%load_ext sql
%sql sqlite:///ecommerce.db

Connecting to 'sqlite:///ecommerce.db'

In [3]:
%%sql
SELECT COUNT(*) AS total_rows
FROM raw_ecommerce_data;

Running query in 'sqlite:///ecommerce.db'

total_rows
112650


In [4]:
%%sql
SELECT
    COUNT(*) AS total_rows,
    SUM(product_category IS NULL) AS null_category,
    SUM(order_delivered_customer_date IS NULL) AS null_delivery_date,
    SUM(review_score IS NULL) AS null_review_score
FROM raw_ecommerce_data;

Running query in 'sqlite:///ecommerce.db'

total_rows,null_category,null_delivery_date,null_review_score
112650,1627,2454,942


In [5]:
%%sql
SELECT
    SUM(product_category IS NULL AND product_name_lenght IS NULL)     AS both_null,
    SUM(product_category IS NULL AND product_name_lenght IS NOT NULL) AS only_category_null,
    SUM(product_category IS NOT NULL AND product_name_lenght IS NULL) AS only_name_null
FROM raw_ecommerce_data;

Running query in 'sqlite:///ecommerce.db'

both_null,only_category_null,only_name_null
1603,24,0


In [6]:
%%sql
SELECT
    product_category AS before,
    COALESCE(NULLIF(TRIM(product_category), ''), "unknown") AS after
FROM raw_ecommerce_data
WHERE product_category IS NULL
LIMIT 5;

Running query in 'sqlite:///ecommerce.db'

before,after
None,unknown
None,unknown
None,unknown
None,unknown
None,unknown


In [7]:
%%sql
SELECT
    COALESCE(NULLIF(LOWER(TRIM(product_category)), ''), 'unknown') AS clean_category,
    COUNT(*) AS n
FROM raw_ecommerce_data
GROUP BY 1
HAVING clean_category = 'unknown';

Running query in 'sqlite:///ecommerce.db'

clean_category,n
unknown,1627


In [8]:
%%sql
SELECT
    order_purchase_timestamp                   AS original,
    CAST(order_purchase_timestamp AS DATETIME) AS year,
    datetime(order_purchase_timestamp)         AS datetime
FROM raw_ecommerce_data
LIMIT 3;

Running query in 'sqlite:///ecommerce.db'

original,year,datetime
2017-10-02 10:56:33,2017,2017-10-02 10:56:33
2018-07-24 20:41:37,2018,2018-07-24 20:41:37
2018-08-08 08:38:49,2018,2018-08-08 08:38:49


In [9]:
%%sql
SELECT
    order_status,
    COUNT(*) AS n,
    SUM(order_delivered_customer_date IS NULL) AS no_delivery_date,
    SUM(order_delivered_customer_date IS NOT NULL) AS has_delivery_date
FROM raw_ecommerce_data
GROUP BY order_status
ORDER BY n DESC;

Running query in 'sqlite:///ecommerce.db'

order_status,n,no_delivery_date,has_delivery_date
delivered,110197,8,110189
shipped,1185,1185,0
canceled,542,535,7
invoiced,359,359,0
processing,357,357,0
unavailable,7,7,0
approved,3,3,0


In [10]:
%%sql
SELECT
    order_id,
    order_status,
    order_purchase_timestamp,
    order_estimated_delivery_date,
    order_delivered_customer_date
FROM raw_ecommerce_data
WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NULL;

Running query in 'sqlite:///ecommerce.db'

order_id,order_status,order_purchase_timestamp,order_estimated_delivery_date,order_delivered_customer_date
2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-12-18 00:00:00,None
f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-07-16 00:00:00,None
2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,2018-07-30 00:00:00,None
e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,2018-07-30 00:00:00,None
0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,2018-07-24 00:00:00,None
2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,2017-06-23 00:00:00,None
ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,2018-06-26 00:00:00,None
20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,2018-07-19 00:00:00,None


In [11]:
%%sql
SELECT
    SUM(CASE WHEN order_status = 'delivered' AND order_delivered_customer_date IS NULL THEN 1 ELSE 0 END) AS flag_delivered_no_date,
    SUM(CASE WHEN order_status <> 'delivered' AND order_delivered_customer_date IS NOT NULL THEN 1 ELSE 0 END) AS flag_date_without_delivery
FROM raw_ecommerce_data;

Running query in 'sqlite:///ecommerce.db'

flag_delivered_no_date,flag_date_without_delivery
8,7


In [12]:
%%sql
SELECT COUNT(*) AS review_before_purchase
FROM raw_ecommerce_data
WHERE review_creation_date < order_purchase_timestamp;

Running query in 'sqlite:///ecommerce.db'

review_before_purchase
5


In [13]:
%%sql
SELECT
    review_comment_message AS before,
    TRIM(REPLACE(REPLACE(review_comment_message, char(13), ' '), char(10), ' ')) AS after
FROM raw_ecommerce_data
WHERE review_comment_message LIKE '%' || char(10) || '%'
LIMIT 3;

Running query in 'sqlite:///ecommerce.db'

before,after
"Lindos. Acabamento incrivel, muiro macios e super em conta.","Lindos. Acabamento incrivel, muiro macios e super em conta."
Somente a Mala Pequena Bordo Avião 360º Viagem Seanite Preta 12771 foi entregue.O Auto Transformador Automático Bivolt 5000va Até 12.000 Btus ainda não foi entregue.,Somente a Mala Pequena Bordo Avião 360º Viagem Seanite Preta 12771 foi entregue. O Auto Transformador Automático Bivolt 5000va Até 12.000 Btus ainda não foi entregue.
Somente a Mala Pequena Bordo Avião 360º Viagem Seanite Preta 12771 foi entregue.O Auto Transformador Automático Bivolt 5000va Até 12.000 Btus ainda não foi entregue.,Somente a Mala Pequena Bordo Avião 360º Viagem Seanite Preta 12771 foi entregue. O Auto Transformador Automático Bivolt 5000va Até 12.000 Btus ainda não foi entregue.


In [14]:
%%sql
SELECT
    order_id,
    order_item_id,
    COUNT(*) AS occurrences
FROM raw_ecommerce_data
GROUP BY order_id, order_item_id
HAVING COUNT(*) > 1
ORDER BY occurrences DESC
LIMIT 20;

Running query in 'sqlite:///ecommerce.db'

order_id,order_item_id,occurrences


In [15]:
%%sql
SELECT
    SUM(order_id IS NULL OR TRIM(order_id) = '') AS missing_order_id,
    SUM(product_id IS NULL OR TRIM(product_id) = '') AS missing_product_id,
    SUM(order_item_id IS NULL) AS missing_item_id,
    SUM(price IS NULL) AS missing_price,
    SUM(price < 0) AS negative_price,
    SUM(freight_value IS NULL) AS missing_freight,
    SUM(freight_value < 0) AS negative_freight,
    SUM(review_score NOT BETWEEN 1 AND 5) AS invalid_review_score
FROM raw_ecommerce_data;

Running query in 'sqlite:///ecommerce.db'

missing_order_id,missing_product_id,missing_item_id,missing_price,negative_price,missing_freight,negative_freight,invalid_review_score
0,0,0,0,0,0,0,0


In [16]:
%%sql
SELECT
    COUNT(*) AS raw_rows,
    SUM(price >= 0) AS rows_retained,
    SUM(price < 0) AS negative_price_rows,
    SUM(price IS NULL) AS missing_price_rows
FROM raw_ecommerce_data;

Running query in 'sqlite:///ecommerce.db'

raw_rows,rows_retained,negative_price_rows,missing_price_rows
112650,112650,0,0


In [20]:
%%sql
DROP TABLE IF EXISTS cleaned_ecommerce_data;

CREATE TABLE cleaned_ecommerce_data AS
SELECT
    TRIM(order_id)                                   AS order_id,
    CAST(order_item_id AS INTEGER)                   AS order_item_id,
    TRIM(product_id)                                 AS product_id,
    LOWER(TRIM(order_status))                        AS order_status,
    COALESCE(NULLIF(LOWER(TRIM(product_category)), ''), 'unknown') AS product_category,
    CAST(product_name_lenght AS INTEGER)        AS product_name_length,
    CAST(product_description_lenght AS INTEGER) AS product_description_length,
    CAST(product_photos_qty AS INTEGER)         AS product_photos_qty,
    datetime(order_purchase_timestamp)               AS order_purchase_timestamp,
    datetime(order_delivered_customer_date)          AS order_delivered_customer_date,
    datetime(order_estimated_delivery_date)          AS order_estimated_delivery_date,
    ROUND(price, 2)                                  AS price,
    ROUND(freight_value, 2)                          AS freight_value,
    CAST(review_score AS INTEGER)                    AS review_score,
    datetime(review_creation_date)                   AS review_creation_date,
            COALESCE(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(review_comment_title,   char(13), ' '), char(10), ' '), '  ', ' '), '  ', ' '), '  ', ' '), '  ', ' ')), '') AS review_comment_title,
    COALESCE(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(review_comment_message, char(13), ' '), char(10), ' '), '  ', ' '), '  ', ' '), '  ', ' '), '  ', ' ')), '') AS review_comment_message,
    
    -- Your existing flags
    CASE WHEN order_status = 'delivered' AND order_delivered_customer_date IS NULL THEN 1 ELSE 0 END      AS flag_delivered_no_date,
    CASE WHEN order_status <> 'delivered' AND order_delivered_customer_date IS NOT NULL THEN 1 ELSE 0 END AS flag_date_without_delivery,
    CASE WHEN review_creation_date < order_purchase_timestamp THEN 1 ELSE 0 END                           AS flag_review_before_purchase,
    
    -- NEW: Timeline logic flags
    CASE WHEN order_delivered_customer_date < order_purchase_timestamp THEN 1 ELSE 0 END                 AS flag_delivered_before_purchase,
    CASE WHEN order_estimated_delivery_date < order_purchase_timestamp THEN 1 ELSE 0 END                 AS flag_estimated_before_purchase,
    
    -- NEW: Outlier price flags
    CASE WHEN price = 0 THEN 1 ELSE 0 END                                                                AS flag_zero_price,
    CASE WHEN price > 5000 THEN 1 ELSE 0 END                                                             AS flag_extreme_price

FROM raw_ecommerce_data
WHERE price >= 0;

Running query in 'sqlite:///ecommerce.db'

++
||
++
++

In [21]:
%%sql
SELECT
    COUNT(*)                                AS total_rows,
    SUM(product_category = 'unknown')       AS unknown_category,
    SUM(flag_delivered_no_date)             AS f1,
    SUM(flag_date_without_delivery)         AS f2,
    SUM(flag_review_before_purchase)        AS f3,
    SUM(flag_delivered_before_purchase)     AS f4,
    SUM(flag_estimated_before_purchase)     AS f5,
    SUM(flag_zero_price)                    AS f6,
    SUM(flag_extreme_price)                 AS f7,
    SUM(review_comment_message LIKE '%  %') AS double_spaces_left
FROM cleaned_ecommerce_data;

Running query in 'sqlite:///ecommerce.db'

total_rows,unknown_category,f1,f2,f3,f4,f5,f6,f7,double_spaces_left
112650,1627,8,7,5,0,0,0,3,0


In [22]:
%%sql
   SELECT * FROM cleaned_ecommerce_data LIMIT 5;

Running query in 'sqlite:///ecommerce.db'

order_id,order_item_id,product_id,order_status,product_category,product_name_length,product_description_length,product_photos_qty,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,price,freight_value,review_score,review_creation_date,review_comment_title,review_comment_message,flag_delivered_no_date,flag_date_without_delivery,flag_review_before_purchase,flag_delivered_before_purchase,flag_estimated_before_purchase,flag_zero_price,flag_extreme_price
e481f51cbdc54678b7cc49136f2d6af7,1,87285b34884572647811a353c7ac498a,delivered,housewares,40,268,4,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18 00:00:00,29.99,8.72,4,2017-10-11 00:00:00,,"Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se trata de um presente.",0,0,0,0,0,0,0
53cdb2fc8bc7dce0b6741e2150273451,1,595fac2a385ac33a80bd5114aec74eb8,delivered,perfumery,29,178,1,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13 00:00:00,118.7,22.76,4,2018-08-08 00:00:00,Muito boa a loja,Muito bom o produto.,0,0,0,0,0,0,0
47770eb9100c2d0c44946d9cf07ec65d,1,aa4383b373c6aca5d8797843e5594415,delivered,auto,46,232,1,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04 00:00:00,159.9,19.22,5,2018-08-18 00:00:00,,,0,0,0,0,0,0,0
949d5b44dbf5de918fe9c16f97b45f8a,1,d0b61bfb1de832b15ba9d266ca96e5b0,delivered,pet_shop,59,468,3,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15 00:00:00,45.0,27.2,5,2017-12-03 00:00:00,,O produto foi exatamente o que eu esperava e estava descrito no site e chegou bem antes da data prevista.,0,0,0,0,0,0,0
ad21c59c0840e6cb83a9ceb5573f8159,1,65266b2da20d04dbe00c5c2d3bb7859e,delivered,stationery,38,316,4,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26 00:00:00,19.9,8.72,5,2018-02-17 00:00:00,,,0,0,0,0,0,0,0
